In [ ]:
from google.colab import drive
drive.mount('/content/drive')

base = "/content/drive/MyDrive/<sua-pasta>/"  # pasta do Drive com movies.csv e customers_rating.csv

linhas = []
with open(base + "movies.csv", encoding="utf-8", errors="replace") as f:
    for linha in f:
        linha = linha.rstrip("\n")
        if not linha:
            continue
        mid, resto = linha.split(";", 1)
        resto = resto.strip()
        if resto.startswith("(") and resto.endswith(")"):
            resto = resto[1:-1]
        titulo, _, ano = resto.rpartition(", ")
        linhas.append((int(mid), titulo.strip(), ano.strip()))

import pandas as pd
movies = pd.DataFrame(linhas, columns=["movie_id", "title", "year"])
movies["year"] = pd.to_numeric(movies["year"], errors="coerce").astype("Int64")
print("Quantidade de filmes:", len(movies))
print(movies.head())


In [ ]:
import os

saida = base + "netflix_dataset.csv"
if os.path.exists(saida):
    os.remove(saida)

total = 0
for i, parte in enumerate(pd.read_csv(base + "customers_rating.csv", sep=";", chunksize=2_000_000)):
    parte.columns = [c.lower() for c in parte.columns]
    junto = parte.merge(movies, on="movie_id", how="left")
    junto.to_csv(saida, mode="a", header=(i == 0), index=False)
    total += len(junto)
    print("linhas processadas:", total)

print("\nPronto:", saida)
print("Tamanho (MB):", round(os.path.getsize(saida) / 1e6))
print(pd.read_csv(saida, nrows=5))

In [ ]:
import os
print("Tamanho no Drive (MB):", round(os.path.getsize(saida) / 1e6))

In [ ]:
import gzip, shutil, os
from google.colab import drive

local = "/content/netflix_dataset.csv.gz"
total = 0
with gzip.open(local, "wt", encoding="utf-8", newline="") as gz:
    for i, parte in enumerate(pd.read_csv(base + "customers_rating.csv", sep=";", chunksize=2_000_000)):
        parte.columns = [c.lower() for c in parte.columns]
        junto = parte.merge(movies, on="movie_id", how="left")
        junto.to_csv(gz, header=(i == 0), index=False)
        total += len(junto)
        print("linhas processadas:", total)

print("\nTamanho local (MB):", round(os.path.getsize(local) / 1e6))
shutil.copy(local, base + "netflix_dataset.csv.gz")
drive.flush_and_unmount()
print("Copiado para o Drive. Linhas no total:", total)